In [1]:
!pip install pandas

In [2]:
import pandas as pd
import numpy as np

In [21]:
df = pd.read_parquet("./data/interim/train_clean.parquet")

In [22]:
print(df.shape)


(4500000, 17)


In [23]:
# Creating calender features
df['day_of_week'] = df['dt'].dt.dayofweek
df['week_of_year'] = df['dt'].dt.isocalendar().week.astype(int)
df['month'] = df['dt'].dt.month
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print(df[['dt', 'day_of_week', 'week_of_year', 'month', 'is_weekend']].head(7))

          dt  day_of_week  week_of_year  month  is_weekend
0 2024-03-28            3            13      3           0
1 2024-03-29            4            13      3           0
2 2024-03-30            5            13      3           1
3 2024-03-31            6            13      3           1
4 2024-04-01            0            14      4           0
5 2024-04-02            1            14      4           0
6 2024-04-03            2            14      4           0


In [26]:
# Creating per store-product pairs
df = df.sort_values(['store_id','product_id','dt']).reset_index(drop=True)

grp = df.groupby(['store_id','product_id'])['sale_amount']

for lag in [1,7,14,28]:
  df[f'sales_lag_{lag}'] = grp.shift(lag)

In [27]:
print(df[['store_id', 'product_id', 'dt', 'sale_amount',
          'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28']].head(10))

   store_id  product_id         dt  sale_amount  sales_lag_1  sales_lag_7  \
0         0           4 2024-03-28          0.5          NaN          NaN   
1         0           4 2024-03-29          1.3          0.5          NaN   
2         0           4 2024-03-30          5.3          1.3          NaN   
3         0           4 2024-03-31          4.2          5.3          NaN   
4         0           4 2024-04-01          0.7          4.2          NaN   
5         0           4 2024-04-02          2.3          0.7          NaN   
6         0           4 2024-04-03          1.3          2.3          NaN   
7         0           4 2024-04-04          4.7          1.3          0.5   
8         0           4 2024-04-05          3.1          4.7          1.3   
9         0           4 2024-04-06          5.5          3.1          5.3   

   sales_lag_14  sales_lag_28  
0           NaN           NaN  
1           NaN           NaN  
2           NaN           NaN  
3           NaN         

In [28]:
# Rolling features capture the sales trend over the past N days
# We use shift(1) first to avoid leakage - only past data is used, never today

for win in [7,14,28]:
  # Average sales over the last N days
  df[f'rolling_mean_{win}'] = grp.shift(1).transform(
      lambda x : x.rolling(win,min_periods=1).mean()
  )
  # Volatility of sales over last N days
  df[f'rolling_std_{win}'] = grp.shift(1).transform(
        lambda x: x.rolling(win, min_periods=1).std()
    )

In [29]:
# Peak and lowest demand in last 7 days
df['rolling_max_7'] = grp.shift(1).transform(
    lambda x: x.rolling(7, min_periods=1).max()
)
df['rolling_min_7'] = grp.shift(1).transform(
    lambda x: x.rolling(7, min_periods=1).min()
)

In [30]:
print(df[['dt', 'sale_amount', 'rolling_mean_7', 'rolling_std_7',
          'rolling_max_7', 'rolling_min_7']].head(10))

          dt  sale_amount  rolling_mean_7  rolling_std_7  rolling_max_7  \
0 2024-03-28          0.5             NaN            NaN            NaN   
1 2024-03-29          1.3        0.500000            NaN            0.5   
2 2024-03-30          5.3        0.900000       0.565685            1.3   
3 2024-03-31          4.2        2.366667       2.571640            5.3   
4 2024-04-01          0.7        2.825000       2.291106            5.3   
5 2024-04-02          2.3        2.400000       2.200000            5.3   
6 2024-04-03          1.3        2.383333       1.968163            5.3   
7 2024-04-04          4.7        2.228571       1.842746            5.3   
8 2024-04-05          3.1        2.828571       1.869683            5.3   
9 2024-04-06          5.5        3.085714       1.743969            5.3   

   rolling_min_7  
0            NaN  
1            0.5  
2            0.5  
3            0.5  
4            0.5  
5            0.5  
6            0.5  
7            0.5  
8  

In [31]:
# How many stockout days in the last 7 and 28 days
# Helps model learn if this product has been frequently out of stock recently
stk_grp = df.groupby(['store_id','product_id'])['stockout_flag']

In [32]:
df['stockout_lag_1']    = stk_grp.shift(1)
df['stockout_count_7']  = stk_grp.shift(1).transform(
    lambda x: x.rolling(7, min_periods=1).sum()
)

In [33]:
df['stockout_count_28'] = stk_grp.shift(1).transform(
    lambda x: x.rolling(28, min_periods=1).sum()
)

In [34]:
print(df[['dt', 'stockout_flag', 'stockout_lag_1',
          'stockout_count_7', 'stockout_count_28']].head(10))


          dt  stockout_flag  stockout_lag_1  stockout_count_7  \
0 2024-03-28              1             NaN               NaN   
1 2024-03-29              1             1.0               1.0   
2 2024-03-30              0             1.0               2.0   
3 2024-03-31              0             0.0               2.0   
4 2024-04-01              0             0.0               2.0   
5 2024-04-02              0             0.0               2.0   
6 2024-04-03              1             0.0               2.0   
7 2024-04-04              0             1.0               3.0   
8 2024-04-05              0             0.0               2.0   
9 2024-04-06              1             0.0               1.0   

   stockout_count_28  
0                NaN  
1                1.0  
2                2.0  
3                2.0  
4                2.0  
5                2.0  
6                2.0  
7                3.0  
8                3.0  
9                3.0  


In [36]:
before = len(df)
df = df.dropna()
after = len(df)

In [37]:
print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Dropped:     {before - after}")

Rows before: 3100000
Rows after:  3100000
Dropped:     0


In [38]:
print(df[['sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28']].isnull().sum())


sales_lag_1     0
sales_lag_7     0
sales_lag_14    0
sales_lag_28    0
dtype: int64


In [39]:
print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col}")

Shape: (3100000, 36)

Columns (36):
  store_id
  product_id
  city_id
  first_category_id
  second_category_id
  third_category_id
  dt
  sale_amount
  stock_hour6_22_cnt
  stockout_flag
  discount
  holiday_flag
  activity_flag
  precpt
  avg_temperature
  avg_humidity
  avg_wind_level
  day_of_week
  week_of_year
  month
  is_weekend
  sales_lag_1
  sales_lag_7
  sales_lag_14
  sales_lag_28
  rolling_mean_7
  rolling_std_7
  rolling_mean_14
  rolling_std_14
  rolling_mean_28
  rolling_std_28
  rolling_max_7
  rolling_min_7
  stockout_lag_1
  stockout_count_7
  stockout_count_28


In [42]:
df.to_parquet("../data/processed/train_features.parquet", index=False)
print("Saved to data/processed/train_features.parquet")
print(df.shape)